In [79]:
import numpy as np
class IdentityActivator(object):
    def forward(self, weighted_input):
        return weighted_input
    def backward(self, output):
        return 1

#Activator类实现激活函数，其中forward方法实现了前向计算，backward方法则是计算导数
class ReluActivator(object):
    def forward(self,weighted_input):
        return max(0,weighted_input)
    def backward(self,output):  # RELU的导函数
        return 1 if output > 0 else 0

In [80]:
# ConvLayer类中的forward()过程中用到的三个函数conv(),padding(),element_wise_op()。get_patch()是conv()用到的。
def get_patch(input_array, i, j, filter_width,filter_height, stride):
    '''从输入数组中获取本次卷积的区域，自动适配输入为2D和3D的情况'''
    start_i = i * stride
    start_j = j * stride
    if input_array.ndim == 2:  # 如果输入是2维数组（height、weight）
        return input_array[start_i : start_i + filter_height, start_j : start_j + filter_width]
    elif input_array.ndim == 3:  # 如果输入是3维数组（depth、height、width）
        return input_array[:, start_i : start_i + filter_height, start_j : start_j + filter_width]

def conv(input_array,kernel_array,output_array,stride,bias):
    """二维卷积实现（独立函数，处理单通道或多通道输入）"""
    channel_number = input_array.ndim  # 获取输入数组的维度数（2或3）
    # 可以看到，ConvLayer类中在**前向计算**中调用本函数conv时，传入形参output_array的实参是self.output_array[f],在ConvLayer的构造函数中,self.output_array = np.zeros((self.filter_number,self.output_height,self.output_width))，self.output_array[f]的形状是(height,width)
    output_width = output_array.shape[1]
    output_height = output_array.shape[0]
    # 可以看到，ConvLayer类中在**前向计算**中调用本函数conv时，传入形参kernel_array的实参是filter.get_weights(),在Filter类中,get_weights()返回的形状是(depth,height,width)
    kernel_width = kernel_array.shape[-1]
    kernel_height = kernel_array.shape[-2]
    # 1.get_patch提取输入数组上与卷积核大小相同的区域。2.将该区域与卷积核逐元素相乘后求和。3.加上偏置，赋值给 output_array[i][j]
    for i in range(output_height):
        for j in range(output_width):
            output_array[i][j] = (get_patch(input_array,i,j,kernel_width,kernel_height,stride)*kernel_array).sum()+bias

def padding(input_array,zp):
    """为输入增加Zero padding。自动适配2D和3D的情况"""
    if zp == 0 :
        return input_array
    else:
        if input_array.ndim == 3:  # 如果输入是3维数组（depth、height、width）
            input_width = input_array.shape[2]
            input_height = input_array.shape[1]
            input_depth = input_array.shape[0]
            padded_array = np.zeros((input_depth,input_height+2*zp,input_width+2*zp))  # 创建空数组，高度和宽度各增加 2*zp
            padded_array[:,zp:zp+input_height,zp:zp+input_width] = input_array  # 将原始数组放入填充数组的中央区域
            return padded_array
        elif input_array.ndim == 2:  # 如果输入是2维数组（height、weight）
            input_width = input_array.shape[1]
            input_height = input_array.shape[0]
            padded_array = np.zeros((input_height+2*zp,input_width+2*zp))  # 创建空数组，高度和宽度各增加 2*zp
            padded_array[zp:zp+input_height,zp:zp+input_width] = input_array  # 将原始数组放入填充数组的中央区域
            return padded_array

def element_wise_op(array,op):
    """ 对array逐元素应用op操作（激活函数）"""
    for i in np.nditer(array, op_flags=['readwrite']): # 使用 np.nditer 遍历数组，允许读写元素
        # 将当前位置的元素值替换为 op(i) 的结果
        i[...] = op(i)

In [91]:
#一个卷积层
class ConvLayer(object):
    def __init__(self,input_width,input_height,channel_number,filter_width,filter_height,filter_number,zero_padding,stride,activator,learning_rate):
        self.input_width = input_width
        self.input_height = input_height
        self.channel_number = channel_number #输入数据的通道数（例如RGB图像为3）
        
        self.filter_width = filter_width
        self.filter_height = filter_height
        self.filter_number = filter_number #卷积核的数量（即输出特征图的通道数）
        
        self.zero_padding = zero_padding
        self.stride = stride
        
        # 计算输出特征图的宽度和高度
        self.output_width = ConvLayer.calculate_output_size(self.input_width,filter_width,zero_padding,stride)
        self.output_height = ConvLayer.calculate_output_size(self.input_height,filter_height,zero_padding,stride)
        self.output_array = np.zeros((self.filter_number,self.output_height,self.output_width))  # 输出特征图数组

        # 初始化所有卷积核
        self.filters = [] 
        for i in range(filter_number):
            self.filters.append(Filter(filter_width,filter_height,self.channel_number))
            
        self.activator = activator
        self.learning_rate = learning_rate

    # calculate_output_size 函数用来确定卷积层输出的大小
    @staticmethod
    def calculate_output_size(input_size,filter_size,zero_padding,stride):
        return (input_size-filter_size+2*zero_padding)//stride + 1

    #前向计算
    def forward(self,input_array):
        """计算卷积层的输出，输出结果保存在self.output_array"""
        self.input_array = input_array
        self.padded_input_array = padding(input_array,self.zero_padding)
        for f in range(self.filter_number):  # 遍历每一个卷积核
            filter = self.filters[f]  # 取出第f个卷积核对象
            # 调用conv函数，进行卷积运算：输出放在output_array的第f层
            conv(self.padded_input_array,filter.get_weights(),self.output_array[f],self.stride,filter.get_bias())
        element_wise_op(self.output_array,self.activator.forward) #对输出特征图逐元素应用激活函数
            
    #反向传播：1.将误差传递到上一层。2.计算每个参数的梯度。3.更新参数
    def backward(self, input_array, sensitivity_array, activator):
        '''计算传递给前一层的误差项，以及计算每个权重的梯度。 前一层的误差项保存在self.delta_array。 梯度保存在Filter对象的weights_grad'''
        self.forward(input_array)
        self.bp_sensitivity_map(sensitivity_array, activator)
        self.bp_gradient(sensitivity_array)

    def update(self):
        '''按照梯度下降，更新权重'''
        for filter in self.filters:
            filter.update(self.learning_rate)
    
    def bp_sensitivity_map(self,sensitivity_array,activator):
        """sensitivity_array: 本层的灵敏度映射（即损失函数对本层输出的偏导）activator: 上一层的激活函数，用于计算上一层的导数"""
        expanded_array = self.expand_sensitivity_map(sensitivity_array) # 处理卷积步长，对原始sensitivity map进行扩展（在步长大于1时插入零）
        expanded_width = expanded_array.shape[2]  # 获取扩展后的灵敏度映射的宽度
        zp = (self.input_width + self.filter_width - 1 - expanded_width) // 2  # 计算为了得到与输入相同尺寸所需的零填充大小
        padded_array = padding(expanded_array, zp)  # 对扩展后的灵敏度映射进行填充，以便进行互相关运算
        self.delta_array = self.create_delta_array()  # 初始化delta_array（误差项，传递到上一层的梯度）
        for f in range(self.filter_number):    # 遍历每一个卷积核
            filter = self.filters[f]
            # 将filter权重翻转180度
            flipped_weights = np.array(list(map( lambda i: np.rot90(i, 2), filter.get_weights())))
            # 创建一个临时的delta_array，用于累加每个卷积核贡献的误差
            delta_array = self.create_delta_array()
            for d in range(delta_array.shape[0]):  # 遍历输出通道（delta_array.shape[0]为channel_number）
                conv(padded_array[f], flipped_weights[d], delta_array[d], 1, 0) # 对填充后的扩展灵敏度映射和翻转后的权重进行卷积（步长为1），结果存入delta_array的对应通道
            self.delta_array += delta_array  # 累加每个卷积核产生的误差项
        # 计算输入数组（本层输入）的激活函数导数
        derivative_array = np.array(self.input_array)  
        element_wise_op(derivative_array,activator.backward)
        self.delta_array *= derivative_array  # 将误差项与激活函数的导数逐元素相乘，得到最终传递到上一层的灵敏度映射

    def bp_gradient(self, sensitivity_array):
        """计算权重的梯度（用于更新参数）。# sensitivity_array: 本层的灵敏度映射"""
        expanded_array = self.expand_sensitivity_map(sensitivity_array) #处理卷积步长，对原始sensitivity map进行扩展
        for f in range(self.filter_number):  # 遍历每一个卷积核
            # 计算每个权重的梯度
            filter = self.filters[f]
            for d in range(filter.weights.shape[0]):  # 遍历卷积核的每个输入通道（weights.shape[0]是输入通道数）
                conv(self.padded_input_array[d], expanded_array[f], filter.weights_grad[d], 1, 0)  # 对填充后的输入数组的第d通道与扩展后的灵敏度映射的第f层进行卷积（步长为1），结果作为权重的梯度
            filter.bias_grad = expanded_array[f].sum()  # 计算偏置项的梯度：扩展后灵敏度映射中所有元素的和

    def expand_sensitivity_map(self, sensitivity_array):
        """扩展灵敏度映射，处理步长大于1的情况（在步长间隔位置插入零）"""
        depth = sensitivity_array.shape[0]   # 获取灵敏度映射的深度（通道数）
        # 确定扩展后sensitivity map的大小，stride为1
        expanded_width = (self.input_width - self.filter_width + 2 * self.zero_padding + 1)  
        expanded_height = (self.input_height - self.filter_height + 2 * self.zero_padding + 1)
        # 创建全零数组，用于存放扩展后的灵敏度映射
        expand_array = np.zeros((depth, expanded_height, expanded_width))
        # 从原始sensitivity map拷贝误差值到扩展后的对应位置
        for i in range(self.output_height):
            for j in range(self.output_width):
                i_pos = i * self.stride
                j_pos = j * self.stride
                expand_array[:,i_pos,j_pos] = sensitivity_array[:,i,j]
        return expand_array

    def create_delta_array(self):
        """# 创建一个全零的delta数组，形状与输入相同（通道数×输入高×输入宽）"""
        return np.zeros((self.channel_number,self.input_height, self.input_width))

In [92]:
#Filter类保存了卷积层的参数以及梯度，并且实现了用梯度下降算法来更新参数
class Filter(object):
    def __init__(self,width,height,depth):
        self.weights = np.random.uniform(-1e-4,1e-4,(depth,height,width))
        self.bias = 0
        self.weights_grad = np.zeros(self.weights.shape)
        self.bias_grad = 0

    def __repr__(self): # NumPy 数组的完整表示，它会展示数组的形状、维度和所有元素
        return 'filter weights:\n%s \n bias:\n%s' % (repr(self.weights),repr(self.bias))

    def get_weights(self):
        return self.weights

    def get_bias(self):
        return self.bias

    def update(self,learning_rate):
        self.weights -= learning_rate*self.weights_grad
        self.bias -= learning_rate*self.bias_grad

In [93]:
# Max Pooling层的实现
class MaxPoolingLayer(object):
    def __init__(self,input_width,input_height,channel_number,filter_width,filter_height,stride):
        self.input_width = input_width
        self.input_height = input_height
        self.channel_number = channel_number
        self.filter_width = filter_width
        self.filter_height = filter_height
        self.stride = stride
        self.output_width = (input_width - filter_width)/self.stride + 1
        self.output_height = (input_height - filter_height)/self.stride + 1
        self.output_array = np.zeros((self.channel_number,self.output_height,self.output_width))

    # 前向传播
    def forward(self,input_array):
        for d in range(self.channel_number):
            for i in range(self.output_height):
                for j in range(self.output_width):
                    self.output_array[d,i,j] = (get_patch(input_array[d],i,j,self.filter_width,self.filter_height,self.stride).max())

    # 反向传播
    def get_max_index(array):
        max_i = 0
        max_j = 0
        max_value = array[0,0]
        for i in range(array.shape[0]):
            for j in range(array.shape[1]):
                if array[i,j] > max_value:
                    max_value = array[i,j]
                    max_i, max_j = i, j
        return max_i, max_j
    
    def backward(self,input_array,sensitivity_array):
        self.delta_array = np.zeros(input_array.shape)
        for d in range(self.channel_number):
            for i in range(self.output_height):
                for j in range(self.output_width):
                    patch_array = get_patch(input_array[d],i,j,self.filter_width,self.filter_height,self.stride)
                    k,l = get_max_index(patch_array)
                    self.delta_array[d,i*self.stride+k,j*self.stride+l] = sensitivity_array[d,i,j]

In [94]:
def init_test():
    a = np.array(
        [[[0,1,1,0,2],
          [2,2,2,2,1],
          [1,0,0,2,0],
          [0,1,1,0,0],
          [1,2,0,0,2]],
         [[1,0,2,2,0],
          [0,0,0,2,0],
          [1,2,1,2,1],
          [1,0,0,0,0],
          [1,2,1,1,1]],
         [[2,1,2,0,0],
          [1,0,0,1,0],
          [0,2,1,0,1],
          [0,1,2,2,2],
          [2,1,0,0,1]]])
    b = np.array(
        [[[0,1,1],
          [2,2,2],
          [1,0,0]],
         [[1,0,2],
          [0,0,0],
          [1,2,1]]])
    cl = ConvLayer(5,5,3,3,3,2,1,2,IdentityActivator(),0.001)
    cl.filters[0].weights = np.array(
        [[[-1,1,0],
          [0,1,0],
          [0,1,1]],
         [[-1,-1,0],
          [0,0,0],
          [0,-1,0]],
         [[0,0,-1],
          [0,1,0],
          [1,-1,-1]]], dtype=np.float64)
    cl.filters[0].bias=1
    cl.filters[1].weights = np.array(
        [[[1,1,-1],
          [-1,-1,1],
          [0,-1,1]],
         [[0,1,0],
         [-1,0,-1],
          [-1,1,0]],
         [[-1,0,0],
          [-1,0,1],
          [-1,0,0]]], dtype=np.float64)
    return a, b, cl


def test():
    a, b, cl = init_test()
    cl.forward(a)
    print(cl.output_array)

def test_bp():
    a, b, cl = init_test()
    cl.backward(a, b, IdentityActivator())
    cl.update()
    print(cl.filters[0])
    print(cl.filters[1])


def gradient_check():
    '''梯度检查'''
    # 设计一个误差函数，取所有节点输出项之和
    error_function = lambda o: o.sum()
    # 计算forward值
    a, b, cl = init_test()
    cl.forward(a)
    # 求取sensitivity map
    sensitivity_array = np.ones(cl.output_array.shape, dtype=np.float64)
    # 计算梯度
    cl.backward(a, sensitivity_array, IdentityActivator())
    # 检查梯度
    epsilon = 10e-4
    for d in range(cl.filters[0].weights_grad.shape[0]):
        for i in range(cl.filters[0].weights_grad.shape[1]):
            for j in range(cl.filters[0].weights_grad.shape[2]):
                cl.filters[0].weights[d,i,j] += epsilon
                cl.forward(a)
                err1 = error_function(cl.output_array)
                cl.filters[0].weights[d,i,j] -= 2*epsilon
                cl.forward(a)
                err2 = error_function(cl.output_array)
                expect_grad = (err1 - err2) / (2 * epsilon)
                cl.filters[0].weights[d,i,j] += epsilon
                print('weights(%d,%d,%d): expected - actural %f - %f' % (d, i, j, expect_grad, cl.filters[0].weights_grad[d,i,j]))

def init_pool_test():
    a = np.array(
        [[[1,1,2,4],
          [5,6,7,8],
          [3,2,1,0],
          [1,2,3,4]],
         [[0,1,2,3],
          [4,5,6,7],
          [8,9,0,1],
          [3,4,5,6]]], dtype=np.float64)
    b = np.array(
        [[[1,2],
          [2,4]],
         [[3,5],
          [8,2]]], dtype=np.float64)
    mpl = MaxPoolingLayer(4,4,2,2,2,2)
    return a, b, mpl

def test_pool():
    a, b, mpl = init_pool_test()
    mpl.forward(a)
    print('input array:\n%s\noutput array:\n%s' % (a, mpl.output_array))

def test_pool_bp():
    a, b, mpl = init_pool_test()
    mpl.backward(a, b)
    print('input array:\n%s\nsensitivity array:\n%s\ndelta array:\n%s' % (a, b, mpl.delta_array))

In [95]:
gradient_check()

weights(0,0,0): expected - actural 5.000000 - 5.000000
weights(0,0,1): expected - actural 6.000000 - 6.000000
weights(0,0,2): expected - actural 5.000000 - 5.000000
weights(0,1,0): expected - actural 5.000000 - 5.000000
weights(0,1,1): expected - actural 7.000000 - 7.000000
weights(0,1,2): expected - actural 5.000000 - 5.000000
weights(0,2,0): expected - actural 5.000000 - 5.000000
weights(0,2,1): expected - actural 6.000000 - 6.000000
weights(0,2,2): expected - actural 5.000000 - 5.000000
weights(1,0,0): expected - actural 2.000000 - 2.000000
weights(1,0,1): expected - actural 1.000000 - 1.000000
weights(1,0,2): expected - actural 2.000000 - 2.000000
weights(1,1,0): expected - actural 9.000000 - 9.000000
weights(1,1,1): expected - actural 9.000000 - 9.000000
weights(1,1,2): expected - actural 9.000000 - 9.000000
weights(1,2,0): expected - actural 2.000000 - 2.000000
weights(1,2,1): expected - actural 1.000000 - 1.000000
weights(1,2,2): expected - actural 2.000000 - 2.000000
weights(2,